### SenseToAct

Python port of `notebooks/ts/SenseToAct.ipynb`. This notebook uses the shared `max_plus.py` operators and writes generated visualization HTML under `notebooks/html/`.

<img src="../../img/SenseToAct.svg" alt="drawing" width="500"/>

In [ ]:
import numpy as np

from max_plus import EPS, INF, oplus, otimes, reset
from timeline import write_timeline_to_file

print("Max-plus algebra and timeline helpers imported successfully")

In [ ]:
# System Max-Plus Encoding
# System setup for Sensor-Processor-Actuator pipeline.

# Physical firing times: [sensor, processor, actuator]
x = np.array([[EPS, EPS, EPS]])

# Single logical time variable, then "blast" to a 1x3 vector.
t = EPS
t_bar = np.array([[t, t, t]])

# Indices for system components.
sensor = 0
processor = 1
actuator = 2

# Execution times: [e_sensor, e_processor, e_actuator]
e_diag = np.array([[0.1, 0.1, 0.1]])

# Sensor period.
sensor_period = 1

print("System setup complete")

In [ ]:
# Evolution equations for system in factored form:
# x' = (GammaStar otimes B) otimes x oplus (GammaStar otimes tBar)
# GammaStar: transitive closure of reaction dependencies.
# B: diagonal matrix of execution times.

gamma = np.array([
    [0, EPS, EPS],
    [e_diag[0][sensor], 0, EPS],
    [EPS, e_diag[0][processor], 0],
])

gamma_star = otimes(gamma, gamma)

# B is diagonal with execution times on the diagonal.
B = np.array([
    [e_diag[0][sensor], EPS, EPS],
    [EPS, e_diag[0][processor], EPS],
    [EPS, EPS, e_diag[0][actuator]],
])

print("Gamma:", gamma)
print("GammaStar:", gamma_star)
print("B (diagonal):", B)

In [ ]:
def step(x, t_bar):
    x_t = x.T
    t_t = t_bar.T

    # (GammaStar otimes B) otimes x
    GB = otimes(gamma_star, B)
    GBx = otimes(GB, x_t)

    # GammaStar otimes tBar
    Gt = otimes(gamma_star, t_t)

    # Combine and transpose back to row vector.
    return oplus(GBx, Gt).T


def report_stats(k, x, t):
    lag_k = x[0] - t
    print(f"Tag index k={k}")
    print(f"t(k)= {t}")
    print(f"x(k) = earliest possible firing times = {x[0]}")
    print(f"lag(k) = {lag_k}")


print("System functions defined")

In [ ]:
# System Simulation Setup

t = 0
t_bar = np.array([[t, t, t]])

# Reset x to eps using the shared helper.
x = reset(x)

k = 0

print("System simulation ready")

In [ ]:
# Execute one simulation step.
x = step(x, t_bar)
k += 1

report_stats(k, x, t)

In [ ]:
# Execute repeatedly.
t = t + sensor_period
t_bar = np.array([[t, t, t]])

x = step(x, t_bar)
k += 1

report_stats(k, x, t)

## HTML Timeline Export

Run a fresh multi-step simulation for visualization and write the generated HTML under `notebooks/html/`.

In [ ]:
timeline_x = reset(x)
timeline_t = 0

x_history = []
t_history = []
e_history = []

for tag_index in range(20):
    if tag_index > 0:
        timeline_t += sensor_period

    timeline_t_bar = np.array([[timeline_t, timeline_t, timeline_t]])
    timeline_x = step(timeline_x, timeline_t_bar)

    x_history.append(timeline_x.copy())
    t_history.append(timeline_t)
    e_history.append(e_diag[0].copy())

write_timeline_to_file(
    x_history,
    t_history,
    e_history,
    reaction_names=["Sensor", "Processor", "Actuator"],
    filename="../html/sense-to-act-timeline.html",
    title="SenseToAct Timeline",
    subtitle="Earliest reaction firing times for the SenseToAct max-plus model.",
)